셀레니움을 활용한 동적 크롤링

In [9]:
from selenium import webdriver

browser = webdriver.Chrome()

url = 'http://naver.com'

browser.get(url)

In [10]:
# 1. 셀레니움과 By를 먼저 가져옴
from selenium import webdriver
from selenium.webdriver.common.by import By

# 2. 크롬 브라우저를 실제로 켜기
browser = webdriver.Chrome()

# 3. 웹사이트로 이동 (켜진 브라우저가 제어됨)
browser.get("https://www.naver.com")

# 4. 검색창을 찾기
elem = browser.find_element(By.ID, 'query')

In [ ]:
# (참고) 실무에 중요한 문법 제미나이로 뽑아본 것


import time
import csv
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
# 실무 필수: 특정 요소가 뜰 때까지 컴퓨터를 '스마트하게' 대기시키는 옵션
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ==========================================
# 1. 브라우저 초기화 및 시작 옵션 설정
# ==========================================
options = webdriver.ChromeOptions()
# options.add_argument('--headless') # 실무 팁: 창을 띄우지 않고 백그라운드에서 몰래 돌릴 때 주석 해제

browser = webdriver.Chrome(options=options)
browser.maximize_window() # 실무 팁: 창이 작으면 버튼이 숨겨져서 에러가 나므로 무조건 창을 키우고 시작!

try:
    # ==========================================
    # 2. 웹페이지 이동 및 스마트 대기 (Implicit / Explicit Wait)
    # ==========================================
    browser.get("https://www.naver.com")
    
    # [실무 필수] 무조건 time.sleep을 쓰면 느려지므로, 
    # "최대 10초 기다리되, ID가 query인 검색창이 화면에 뜨면 즉시 다음 코드로 넘어가라"는 스마트 대기 구문입니다.
    search_box = WebDriverWait(browser, 10).until(
        EC.presence_of_element_located((By.ID, "query"))
    )

    # ==========================================
    # 3. 키보드 입력 및 제어 (글자 입력, 엔터, 텍스트 지우기)
    # ==========================================
    search_box.send_keys("아이폰 16") # 검색어 입력
    time.sleep(1) # 사람인 척 하려고 중간중간 일부러 1초씩 쉬어줌 (차단 방지)
    
    # search_box.clear() # 실무 팁: 만약 기존 글자를 지우고 새로 쓰고 싶다면 이 함수 사용
    search_box.send_keys(Keys.ENTER) # 키보드 엔터 키 입력
    time.sleep(3) # 검색 결과 로딩 대기

    # ==========================================
    # 4. 마우스 클릭 및 탭(Tab) 전환 제어
    # ==========================================
    # 검색 결과 중 '쇼핑' 탭 클릭
    shopping_btn = browser.find_element(By.LINK_TEXT, "쇼핑")
    shopping_btn.click()
    time.sleep(3)

    # [실무 필수] 셀레니움은 쇼핑 버튼을 눌러서 '새 창(새 탭)'이 뜨더라도, 
    # 제어권은 여전히 첫 번째 창에 머물러 있습니다. 새 창으로 제어권을 넘겨주는 코드입니다.
    all_windows = browser.window_handles # 현재 켜져 있는 모든 창들의 리스트를 가져옴
    browser.switch_to.window(all_windows[-1]) # 맨 마지막에 열린(가장 최신) 새 창으로 이사 가기!
    print("현재 창 제목:", browser.title) # 새 창으로 잘 이사 갔는지 확인인용

    # ==========================================
    # 5. 무한 스크롤 다운 (자바스크립트 실행하기)
    # ==========================================
    # 인스타그램, 네이버 쇼핑처럼 스크롤을 내려야 데이터가 더 나오는 곳에서 쓰는 실무 기술입니다.
    # 현재 웹페이지의 높이를 구함
    last_height = browser.execute_script("return document.body.scrollHeight")
    
    for i in range(3): # 예시로 아래로 3번만 스크롤을 내립니다.
        # 브라우저에게 자바스크립트 명령어를 내려서 스크롤을 맨 아래로 쾅 내림
        browser.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2) # 스크롤 후 새 상품이 로딩될 때까지 대기
        
        # 스크롤 내린 후의 새로운 높이 계산
        new_height = browser.execute_script("return document.body.scrollHeight")
        if new_height == last_height: # 더 이상 내려갈 곳이 없다면 반복문 탈출
            break
        last_height = new_height

    # ==========================================
    # 6. 복수 원소 추출 (find_elements) 및 데이터 정제
    # ==========================================
    # 1개가 아니라 여러 개의 상품들을 한 번에 뭉텅이로 가져올 때는 뒤에 's'를 붙여 elements를 씁니다.
    # 복수 원소를 찾으면 리스트 형식으로 저장됩니다.
    products = browser.find_elements(By.CSS_SELECTOR, "a[class*='product_title']")
    
    print(f"--- 총 {len(products)}개의 상품을 발견했습니다 ---")
    
    # 획득한 데이터를 담을 빈 리스트 주머니
    data_list = []

    for index, product in enumerate(products[:10], start=1): # 상위 10개만 샘플로 가공
        # 상품의 순수한 글자(텍스트) 추출
        title = product.text
        
        # 태그 안에 숨겨진 속성값(여기서는 상품 클릭 시 이동할 링크 주소) 추출
        link = product.get_attribute('href')
        
        # 실무 팁: 가끔 텍스트가 비어있거나 광고 상품이 섞여서 에러가 날 수 있으므로 예외처리 방어벽 탑재
        if title:
            cleaned_title = title.strip().replace("\n", " ") # 줄바꿈 글자 공백으로 청소
            print(f"[{index}위] {cleaned_title}")
            
            # 리스트에 정제된 데이터를 묶어서 담기
            data_list.append([index, cleaned_title, link])

    # ==========================================
    # 7. 크롤링한 데이터 CSV 파일로 영구 저장하기
    # ==========================================
    filename = '네이버쇼핑_실무크롤링_결과.csv'
    with open(filename, 'w', encoding='utf-8-sig', newline='') as f:
        writer = csv.writer(f)
        
        # 엑셀 맨 첫 줄에 헤더(제목 행) 적기
        writer.writerow(['순위', '상품명', '링크주소'])
        
        # 모아놓은 덩어리 데이터들을 한 번에 통째로 엑셀에 쏟아붓기!
        writer.writerows(data_list)
        
    print(f"🎉 성공적으로 '{filename}' 파일에 저장이 완료되었습니다!")

except Exception as e:
    # 코드가 돌다가 중간에 삐끗해서 에러가 나면 어떤 에러인지 화면에 알려주는 안전장치
    print("❌ 실행 중 에러가 발생했습니다 :", e)

finally:
    # ==========================================
    # 8. 자원 반납 및 안전 종료
    # ==========================================
    time.sleep(5)
    browser.quit() # 에러가 나든 안 나든 브라우저는 메모리를 위해 깔끔하게 닫아줍니다.
    print("브라우저가 안전하게 종료되었습니다.")

In [17]:
# 다음 사이트를 자동으로 로그인하기

# 1. 필요한 라이브러리 불러오기
from selenium import webdriver
from selenium.webdriver.common.by import By   # 클릭을 하기 위해 설정

# 2. 접속할 카카오/다음 로그인 페이지 주소 설정
url = 'https://accounts.kakao.com/login/?continue=https%3A%2F%2Fkauth.kakao.com%2Foauth%2Fauthorize%3Fclient_id%3D53e566aa17534bc816eb1b5d8f7415ee%26prompt%3Dselect_account%26state%3D776dddda-90b4-4b55-a31a-6edce8e25ac7%26redirect_uri%3Dhttps%253A%252F%252Flogins.daum.net%252Faccounts%252Foauth%252Fkakao%252Fcallback%26response_type%3Dcode%26auth_tran_id%3DlR4BHT6ZgzfwTNKv8N78KiztwrJuvcdKoVjWde~MtYe6IoPBK5gjn.fLAYuD%26ka%3Dsdk%252F2.7.6%2520os%252Fjavascript%2520sdk_type%252Fjavascript%2520lang%252Fko-KR%2520device%252FMacIntel%2520origin%252Fhttps%25253A%25252F%25252Flogins.daum.net%2520app_key%252F53e566aa17534bc816eb1b5d8f7415ee%26is_popup%3Dfalse%26through_account%3Dtrue#login'

# 3. 브라우저 실행 및 사이트 접속
browser = webdriver.Chrome()   # 내 컴퓨터에 실제 크롬 브라우저 창을 생성
browser.get(url)   # 위에서 지정한 카카오 로그인 페이지로 이동

my_id = input('아이디를 입력하시오: ')
my_pw = input('비밀번호를 입력하시오: ')

# 로그인 아이디 입력창 찾기
elem_id = browser.find_element(By.ID, 'loginId--1')
elem_id.send_keys(my_id)

# 로그인 비밀번호 입력창 찾기
elem_pw = browser.find_element(By.ID, 'password--2')
elem_pw.send_keys(my_pw)

# 로그인 버튼 찾기
elem_id=browser.find_element(By.XPATH, '//*[@id="mainContent"]/div/div/form/div[4]/button[1]')
elem_id.click()


아이디를 입력하시오:  aaaa
비밀번호를 입력하시오:  bbbb


In [37]:
# 네이버 항공권

from selenium import webdriver
from selenium.webdriver.common.by import By 

browser = webdriver.Chrome()   # 크롬 브라우저 창을 새로 실행
browser.maximize_window()      # 실행된 브라우저 창을 전체 화면으로 극대화

url = 'https://flight.naver.com/'
browser.get(url)   # # 네이버 항공권 사이트로 이동

# 클래스 이름으로 팝업창의 '닫기' 버튼 요소를 탐색
elem = browser.find_element(By.CLASS_NAME, 'FullscreenPopup_close')   # 팝업창이 화면에 완전히 뜰 때까지 2초간 대기
time.sleep(2)   # 찾은 '닫기' 버튼을 마우스로 클릭하여 팝업창 제거
elem.click()


# 가는 날 버튼 선택 -> 캘린더가 뜰 수 있도록
elem = browser.find_element(By.XPATH, '//*[@id="__next"]/div/main/div[3]/div/div/div[2]/div[2]/button[1]')
elem.click()
time.sleep(1) 


# 가는 날 날짜 선택
# # '26'이라는 텍스트를 가진 <b> 태그가 속한 <button> 요소를 '모두' 찾아 리스트로 반환
elems = browser.find_elements(By.XPATH, '//button[b[text()="26"]]')

## elems(리스트)에서 버튼들을 하나씩 꺼내되, 몇 번째 버튼인지 번호(i)를 자동으로 매겨주는 함수
# for i, btn in enumerate(elems):
#     print(i, btn.text)

# 이번 달 26일 선택
browser.find_elements(By.XPATH, '//button[b[text()="26"]]')[0].click()
# 이번 달 28일 선택
time.sleep(1)
browser.find_elements(By.XPATH, '//button[b[text()="28"]]')[0].click()
time.sleep(1)


browser.execute_script("window.scrollTo(0, 1000);") 
time.sleep(1)

# 인기 항공권 중 제주 선택
browser.find_element(By.XPATH, '//*[@id="__next"]/div/main/div[8]/div/div[3]/div[1]/ul/li[1]/div').click()
time.sleep(1)

# browser.find_element(By.XPATH, '//i[text()="제주"] | //b[text()="제주"] | //span[text()="제주"] | //em[text()="제주"]').click()

# 검색 버튼 선택
browser.find_element(By.XPATH, '//*[@id="__next"]/div/main/div[3]/div/div/div[2]/div[4]/button').click()
time.sleep(3)


#html 코드가 생성됨
html = browser.page_source   # 현재 브라우저 창에 열린 웹페이지의 전체 HTML 소스코드를 통째로 가져옴
from bs4 import BeautifulSoup
soup = BeautifulSoup(html, 'lxml')   # 가져온 HTML 텍스트를 파이썬이 분석하기 좋은 뷰티풀수프(Soup) 객체로 변환
print(soup.title)


<title>네이버 항공권</title>


In [77]:
# 간단한 접속이 가능한 페이지 - requests 모듈로 가능함

import requests
from bs4 import BeautifulSoup
import re


url = 'https://school.cbe.go.kr/cbs-h/M01050705/list'

res = requests.get(url)   # # 지정한 웹사이트 주소(url)에 접속하여 페이지의 모든 데이터를 요청하고 받아옴
res.raise_for_status()   # 접속 결과에 에러(예: 404, 500)가 있으면 파이썬 프로그램을 즉시 멈추는 장치
print('응답코드: ', res.status_code)

html = res.text
soup = BeautifulSoup(html, 'lxml')   # 가져온 HTML 텍스트를 파이썬이 분석하기 좋은 뷰티풀수프(Soup) 객체로 변환
# print(soup.title)

# 오늘의 식단 정보 출력

table = soup.find('table', attrs = {'class' : 'tch-sch-tbl'})

days = table.find_all('td', attrs = {'class': 'tue tch-has'})

for day in days:
    day_num = day.find('span').get_text(strip=True)
    print(f"2026년 6월 {day_num}일 급식 메뉴")

    meals = day.find_all('dl')

    for meal in meals:
        time = meal.find('dt').get_text(strip=True)
        menu = [li.text.strip() for li in meal.find_all('li')]
        menu_day = ', '.join(menu)
        
        print(f"[{time}] {menu_day}")

    print()

응답코드:  200
2026년 6월 2일 급식 메뉴
[조식] 쌀밥(소), 쇠고기야채죽 (5.6.16), 메추리알조림 (1.5.6.13), 구운김(완), 배추김치 (9), 요거톡초코볼 (2), 시리얼(2종)+우유 (2.5.6)
[중식] 카레라이스 (2.5.6.10.12.13.16.18), 달걀부추국 (1), 매콤콩나물무침 (5), 꼬들단무지, 치커리유자청무침 (13), 칠리깐쇼새우 (1.5.6.9.12.13), 배추김치 (9), 후르츠생과일케이크 (1.2.5.6.11)

2026년 6월 9일 급식 메뉴
[조식] 간장달걀밥 (1.5.6), 우동장국 (1.2.5.6.7.9.13.18), 애배추오이무침 (5.6.13), 햄쏘옥야채전 (1.2.5.6.10.12.15.16), 배추김치 (9), 시리얼(2종)+우유 (2.5.6)
[중식] 혼합잡곡밥 (5), 등뼈감자탕 (5.6.10.13), 연두부/양념장 (5.6), 탕평채 (1.5.6.13.16), 돈도끼떡갈비 (5.6.10.15.16.18), 배추김치 (9), 동글동글한라봉 (13)
[석식] 쌀밥, 김치어묵국 (1.5.6.9), 대패삼겹숙주볶음 (5.6.10.13.18), 찹쌀콩멸치볶음 (5.13), 깍두기 (9), 방울토마토 (12)

2026년 6월 16일 급식 메뉴
[조식] 쌀밥, 건새우아욱된장국 (5.6.9), 달걀찜 (1), 참치고추장볶음 (5.6.13), 배추김치 (9), 시리얼(2종)+우유 (2.5.6)
[중식] 혼합잡곡밥 (5), 닭개장 (15), 꽈리고추메추리알조림 (1.5.6.13), 부추양파무침 (13), 고등어오븐구이 (5.7), 배추김치 (9), 제로요구르트 (2)
[석식] 쌀밥, 쑥갓어묵국 (1.5.6), 애배추나물된장무침 (5.6), 치킨까스/칠리소스 (1.5.6.12.13.15.18), 배추김치 (9), 젤리블리피치

2026년 6월 23일 급식 메뉴
[조식] 쌀밥, 들깨무채국 (5.6), 베이컨숙주볶음 (5.6.10.13.18), 에그랑땡/케첩 (1.2.5.6.10.12.15.16), 배추

In [80]:
import requests
from bs4 import BeautifulSoup

url = 'https://school.cbe.go.kr/cbs-h/M01050705/list'
res = requests.get(url)
soup = BeautifulSoup(res.text, 'lxml')

# 1. 식단표 테이블에서 모든 날짜 칸(td)을 다 가져옴
# (요일 문제를 피하기 위해 'td' 태그를 전부 다 가져온 뒤, 알맹이가 있는 칸만 골라냄)
all_tds = soup.find_all('td')

for day in all_tds:
    # 칸 안에 식단(dl 태그)이 들어있는 평일 칸만
    if day.find('dl'): 
        
        # 날짜 추출
        day_num = day.find('span').get_text(strip=True)
        print(f"=== 2026년 6월 {day_num}일 급식 메뉴 ===")
        
        # 2. 이 날짜 칸 안에 들어있는 조식/중식/석식 (dl)
        meals = day.find_all('dl')
        
        for meal in meals:
            # "조식", "중식", "석식" 글자 가져오기
            time_type = meal.find('dt').get_text(strip=True)
            
            # 3. 이 식단 안에 들어있는 모든 반찬 글자(li)들을 기본 리스트로 묶기
            menu_list = []
            for li in meal.find_all('li'):
                menu_list.append(li.get_text(strip=True))
            
            # 4. 리스트를 하나의 문자열로 이어 붙이기
            menu_string = ', '.join(menu_list)
            
            print(f"[{time_type}] {menu_string}")
            
        print() # 날짜별 줄바꿈
    

=== 2026년 6월 1일 급식 메뉴 ===
[조식] 쌀밥, 들깨수제비국 (5.6), 고추장멸치볶음 (5.6.13), 심쿵햄 (1.2.5.6.10.12.15.16), 배추김치 (9), 시리얼(2종)+우유 (2.5.6)
[중식] 혼합잡곡밥 (5), 돈육김치찌개 (5.6.9.10), 치즈달걀찜 (1.2), 애배추나물된장무침 (5.6), 데리야끼소시지떡볶음 (2.5.6.10.12.13.15.16), 구운김(완), 깍두기 (9), 노티드퍼플팜 (13)
[석식] 굴소스새우살볶음밥 (1.2.5.6.9.10.13.15.16.18), 팽이된장국 (5.6), 참나물무침, 나쵸치킨 (1.5.6.15.18), 배추김치 (9), 설레임초코 (1.2.5)

=== 2026년 6월 2일 급식 메뉴 ===
[조식] 쌀밥(소), 쇠고기야채죽 (5.6.16), 메추리알조림 (1.5.6.13), 구운김(완), 배추김치 (9), 요거톡초코볼 (2), 시리얼(2종)+우유 (2.5.6)
[중식] 카레라이스 (2.5.6.10.12.13.16.18), 달걀부추국 (1), 매콤콩나물무침 (5), 꼬들단무지, 치커리유자청무침 (13), 칠리깐쇼새우 (1.5.6.9.12.13), 배추김치 (9), 후르츠생과일케이크 (1.2.5.6.11)

=== 2026년 6월 4일 급식 메뉴 ===
[조식] 쌀밥(소), 피자토스트 (1.2.5.6.10.12.13.15.16), 옥수수크림스프 (2.5.6.13.16), 구운김(완), 배추김치 (9), 오렌지주스, 시리얼(2종)+우유 (2.5.6)
[중식] 혼합잡곡밥 (5), 쇠고기뭇국 (5.6.16), 꽁치김치무조림 (5.6.9.13), 미나리숙주무침, 볼어묵볶음 (1.5.6.13), 깍두기 (9), 바나나
[석식] 치킨마요덮밥 (1.5.6.13.15.18), 유부장국 (5.6), 망고용과샐러드 (1.2.5.6), 매콤해물볶음우동 (5.6.7.9.13.17.18), 배추김치 (9), 리치캐모마일

=== 2026년 6월 5일 급식 메뉴 ===
[조식] 쌀밥, 우리쌀초코파

In [40]:
# 복잡한 페이지 - requests로 한 경우

import requests
from bs4 import BeautifulSoup

url = 'https://section.blog.naver.com/BlogHome.nhn?directoryNo=0&currentPage=1&groupId=0'

res = requests.get(url)   # # 지정한 웹사이트 주소(url)에 접속하여 페이지의 모든 데이터를 요청하고 받아옴
res.raise_for_status()   # 접속 결과에 에러(예: 404, 500)가 있으면 파이썬 프로그램을 즉시 멈추는 안전장치
print('응답코드: ', res.status_code)

html = res.text
soup = BeautifulSoup(html, 'lxml')   # 가져온 HTML 텍스트를 파이썬이 분석하기 좋은 뷰티풀수프(Soup) 객체로 변환
# print(soup.title)


title = soup.find_all('strong', attrs = {'class' : 'title_post'})   # 접속은 되는데 데이터를 가져오지 못함
print(title)

# 접속은 되는데 데이터를 가져오지 못함 -> 셀레니움 모듈 필요

응답코드:  200
<title>네이버 블로그</title>
[]


In [97]:
# 복잡한 페이지 - 셀레니움 모듈로 가능


from selenium import webdriver
from selenium.webdriver.common.by import By 
from bs4 import BeautifulSoup

browser = webdriver.Chrome()

url = 'https://section.blog.naver.com/ThemePost.naver?directoryNo=30&activeDirectorySeq=4&currentPage=1'
browser.get(url)

html = browser.page_source

soup = BeautifulSoup(html, 'lxml')

blogs = soup.find('section', attrs = {'class':'wrap_thumbnail_post_list'})


titles = blogs.find_all('strong', attrs = {'class':'title_post'})
likes = blogs.find_all('span', attrs = {'class':'like'})


for title, like in zip(titles, likes):
    title_text = title.get_text(strip=True)
    like_count = like.find('em').get_text(strip=True)
    print(f"제목: {title_text} | 공감 수: {like_count}")

제목: 프롬프트엔지니어링 _ AI를 넘어 AX시대, 처음 만나는 바이브 코딩 X MCP | 공감 수: 16
제목: 클린포터, 카페 사장님들이 주목하는 이유 | 공감 수: 18
제목: 피지컬AI 로봇 밸류체인 산업 발전 및 부문별 관련 기업들 | 공감 수: 158
제목: 현대오토에버 주가 폭발 예고? 현대차그룹 6조 피지컬 AI 투자 수혜 전망 | 공감 수: 24
제목: 마샬헤드폰 신작! 마샬 헤드폰 밀톤 ANC (Milton ANC) 출시 코디 및 실사용 리뷰 | 공감 수: 19
제목: 얼리츠 쿨링 넥밴드 가성비 좋은 아이스 스카프 여름에 꼭 써야하는 제품 | 공감 수: 29
제목: <2차 전지> 데이터센터 On-site ESS 시대 개막이 갖는 의미 | 공감 수: 155
제목: 앤트로픽 페이블, 미토스 수출 통제에 관한 생각 (소버린 AI) | 공감 수: 170
제목: 메타가 구독 프로그램을 꺼낸 이유. | 공감 수: 201
제목: 샘 올트먼 방한 취소 이유, 네이버 주가 월요일 향방은? | 공감 수: 55


In [68]:
# 복잡한 페이지 - selenium 모듈로 가능
import requests
from bs4 import BeautifulSoup

url = 'https://section.blog.naver.com/Search/Post.naver?pageNo=1&rangeType=ALL&orderBy=sim&keyword=%EC%9A%94%EB%A6%AC'

from selenium import webdriver
from selenium.webdriver.common.by import By

browser = webdriver.Chrome()
browser.get(url)
html = browser.page_source

# res = requests.get(url)
# print("응답코드: ", res.status_code)
# html = res.text

soup = BeautifulSoup(html, "lxml")
print(soup.title)

blogs = soup.find_all("div", attrs={"class":"item multi_pic"})
# print(blogs[0].find("span").get_text())

for blog in blogs :
    title = blog.find("span", attrs={"class":"title"}).get_text()
    print(title)

<title>네이버 블로그</title>
6월 일주일 가정식 밑반찬 메뉴 만들기 여름 반찬거리 종류 레시피 추천
간장 돼지불고기 양념 레시피 돼지불백 대패삼겹살 요리
단호박스프 만들기 단호박 요리 크림스프 레시피
마늘쫑 비빔밥 레시피 양념장 만들기 마늘쫑요리
데친 마늘쫑무침 마늘쫑 요리 레시피 마늘종무침 만드는법
서울 서촌 흑백요리사 맛집 : 도량 웨이팅 후기
멸치 마늘쫑볶음 레시피 마늘쫑 멸치볶음 요리 레시피
